# Measure, clean up, measure again

BPA tells you what is wrong, VertiPaq tells you where the bytes are. Fix it with code, then run both again.
On one model this is a trick, on fifty models it is a process.

Runs on the Import model: that's where the size statistics are real.

Prerequisites: run 03b (SM_Contoso_Import), and attach the environment with semantic-link-labs.

In [1]:
# Setup: sempy reads and analyzes models, sempy_labs creates and changes them.
# sempy_labs comes from the attached environment, so no %pip install needed.

import re
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
from importlib.metadata import version

print(f"semantic-link-labs {version('semantic-link-labs')}")

WORKSPACE = "Dataminds DEV"
DATASET = "SM_Contoso_Import"

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 32, Finished, Available, Finished, False)

semantic-link-labs 0.17.1


## Measure: the baseline

Two views on the same model. BPA checks the rules, VertiPaq shows the cost.

In [2]:
# Best Practice Analyzer: which modeling rules does this model break?
# This is the 'before' picture we compare against at the end.

labs.run_model_bpa(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 33, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimCustomer'[CustomerKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimDate'[Day],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimDate'[MonthNumber],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimDate'[Quarter],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimDate'[Year],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimProduct'[ProductKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimProduct'[ProductSubcategoryKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimProductCategory'[ProductCategoryKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimProductSubcategory'[ProductCategoryKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'DimProductSubcategory'[ProductSubcategoryKey],⚠️


In [3]:
# VertiPaq Analyzer: where do the bytes go? Tables, columns, dictionaries, relationships.

labs.vertipaq_analyzer(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 34, Finished, Available, Finished, False)

<sempy_labs.semantic_model._vertipaq_analyzer.visualize_vertipaq.<locals>._VertipaqWidget object at 0x7f65a034e0d0>

{'Model Summary':         Dataset Name  Total Size  Table Count  Column Count  \
 0  SM_Contoso_Import   226714493            8            45   
 
    Compatibility Level Default Mode  
 0                 1702       Import  ,
 'Tables':               Table Name              Type  Row Count  Total Size  Data Size  \
 0        FactOnlineSales             Table   12627608   223074848  101983312   
 1            DimCustomer             Table      18869     3019285     263944   
 2             DimProduct             Table       2517      410562      16752   
 3                DimDate             Table       1096       97570       5064   
 4               DimStore             Table        306       67784       1224   
 5  DimProductSubcategory             Table         44       23210        616   
 6     DimProductCategory             Table          8       19482        400   
 7              _Measures  Calculated Table          1        1752        264   
 
    Dictionary Size  Hierarchy Si

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 35, Finished, Available, Finished, False)

In [4]:
# The ten most expensive columns, with their share of the table and of the model.
# This is where cleanup pays off.

df_size = fabric.list_columns(DATASET, workspace=WORKSPACE, extended=True)

model_total = df_size["Total Size"].sum()
table_totals = df_size.groupby("Table Name")["Total Size"].transform("sum")
df_size["% of Table"] = (df_size["Total Size"] / table_totals * 100).round(2)
df_size["% of Model"] = (df_size["Total Size"] / model_total * 100).round(2)

before_size = model_total
print(f"Model size before cleanup: {before_size / 1024**2:,.0f} MB")

df_size[["Table Name", "Column Name", "Total Size", "Data Size", "Dictionary Size",
         "Column Cardinality", "% of Table", "% of Model"]] \
    .sort_values("Total Size", ascending=False).head(10)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 36, Finished, Available, Finished, False)

Model size before cleanup: 216 MB


,Table Name,Column Name,Total Size,Data Size,Dictionary Size,Column Cardinality,% of Table,% of Model
34,FactOnlineSales,OnlineSalesKey,101021168,50510568,144,12627608,45.29,44.57
39,FactOnlineSales,SalesOrderNumber,85168900,15561184,56213108,1674320,38.19,37.57
38,FactOnlineSales,CustomerKey,11366668,10615640,600036,18869,5.1,5.01
37,FactOnlineSales,ProductKey,7943344,7847296,75872,2516,3.56,3.5
35,FactOnlineSales,DateKey,7108028,7056448,42764,1096,3.19,3.14
41,FactOnlineSales,UnitCost,5214072,5199656,10528,480,2.34,2.3
42,FactOnlineSales,UnitPrice,5205816,5192120,10240,426,2.33,2.3
8,DimCustomer,EmailAddress,910229,37104,725205,18484,30.15,0.4
1,DimCustomer,CustomerKey,788900,37872,600036,18869,26.13,0.35
9,DimCustomer,AddressLine1,654381,37104,514813,12802,21.67,0.29


## Find what nobody uses

A column is in use when it's in a relationship, referenced by a measure, or used as sort-by column.
Everything else is a candidate for removal.

In [5]:
# Collect every column the model actually uses, then list the rest as candidates.
# The big ones at the top are the expensive dead weight.

used = set()

rels = fabric.list_relationships(DATASET, workspace=WORKSPACE)
for _, r in rels.iterrows():
    used.add((r["From Table"], r["From Column"]))
    used.add((r["To Table"], r["To Column"]))

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=True) as tom:
    expressions = " ".join(m.Expression for t in tom.model.Tables for m in t.Measures)
    for t in tom.model.Tables:
        for c in t.Columns:
            if re.search(rf"\[{re.escape(c.Name)}\]", expressions):
                used.add((t.Name, c.Name))
            if c.SortByColumn is not None:
                used.add((t.Name, c.SortByColumn.Name))

df_cand = df_size[
    ~df_size.apply(lambda r: (r["Table Name"], r["Column Name"]) in used, axis=1)
    & ~df_size["Column Name"].str.startswith("RowNumber")
]
df_cand[["Table Name", "Column Name", "Total Size", "Column Cardinality"]] \
    .sort_values("Total Size", ascending=False)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 37, Finished, Available, Finished, False)

,Table Name,Column Name,Total Size,Column Cardinality
34,FactOnlineSales,OnlineSalesKey,101021168,12627608
39,FactOnlineSales,SalesOrderNumber,85168900,1674320
8,DimCustomer,EmailAddress,910229,18484
9,DimCustomer,AddressLine1,654381,12802
10,DimCustomer,Phone,358871,8890
15,DimProduct,ProductName,149067,2517
16,DimProduct,ProductDescription,84385,809
6,DimCustomer,BirthDate,70280,8253
3,DimCustomer,FirstName,61894,670
5,DimCustomer,LastName,49686,375


## Clean up with code

Remove the two expensive fact columns nobody uses, hide the keys,
and stop numeric columns from being summed by accident.

In [6]:
# 1. Remove unused high-cardinality columns: they cost the most and add nothing.
# 2. Hide keys: report builders should use the dimension attributes, not the IDs.
# 3. SummarizeBy None on numeric columns: aggregation belongs in measures.
# Already removed columns are skipped, so the cell can safely run twice.

REMOVE = ["OnlineSalesKey", "SalesOrderNumber"]

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    fact = tom.model.Tables["FactOnlineSales"]
    for col in REMOVE:
        if fact.Columns.Find(col) is not None:
            tom.remove_object(object=fact.Columns[col])
            print(f"Removed: FactOnlineSales[{col}]")

    for t in tom.model.Tables:
        for c in t.Columns:
            if c.Name.endswith("Key"):
                c.IsHidden = True
            if str(c.DataType) in ("Int64", "Decimal", "Double") and not c.Name.startswith("RowNumber"):
                tom.set_summarize_by(table_name=t.Name, column_name=c.Name, value="None")

print("Keys hidden, numeric columns set to SummarizeBy None")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 38, Finished, Available, Finished, False)

Removed: FactOnlineSales[OnlineSalesKey]
Removed: FactOnlineSales[SalesOrderNumber]
Keys hidden, numeric columns set to SummarizeBy None


In [7]:
# Reprocess so VertiPaq can measure the new, smaller model.

labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 39, Finished, Available, Finished, False)

⌛ Refresh of the 'SM_Contoso_Import' semantic model within the 'Dataminds DEV' workspace is in progress...
🟢 Refresh 'full' of the 'SM_Contoso_Import' semantic model within the 'Dataminds DEV' workspace is complete.


## Measure again

In [8]:
# Same measurement as the baseline: the difference is the result of the cleanup.

fabric.refresh_tom_cache(workspace=WORKSPACE)
after_size = fabric.list_columns(DATASET, workspace=WORKSPACE, extended=True)["Total Size"].sum()

print(f"Before: {before_size / 1024**2:,.0f} MB")
print(f"After:  {after_size / 1024**2:,.0f} MB")
print(f"Saved:  {(before_size - after_size) / 1024**2:,.0f} MB ({(1 - after_size / before_size) * 100:,.0f}%)")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 40, Finished, Available, Finished, False)

Before: 216 MB
After:  38 MB
Saved:  178 MB (82%)


In [9]:
# BPA again: the hidden keys and summarization warnings should be gone.

labs.run_model_bpa(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 41, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
Mark primary keysSet the 'Key' property to 'True' for primary key columns within the column properties.,Column,'DimCustomer'[CustomerKey],ℹ️
Mark primary keysSet the 'Key' property to 'True' for primary key columns within the column properties.,Column,'DimProduct'[ProductKey],ℹ️
Mark primary keysSet the 'Key' property to 'True' for primary key columns within the column properties.,Column,'DimProductCategory'[ProductCategoryKey],ℹ️
Mark primary keysSet the 'Key' property to 'True' for primary key columns within the column properties.,Column,'DimProductSubcategory'[ProductSubcategoryKey],ℹ️
Mark primary keysSet the 'Key' property to 'True' for primary key columns within the column properties.,Column,'DimStore'[StoreKey],ℹ️
"Percentages should be formatted with thousands separators and 1 decimalFor a better user experience, percengage measures should be formatted with a '%' sign.",Measure,Margin %,⚠️
"Percentages should be formatted with thousands separators and 1 decimalFor a better user experience, percengage measures should be formatted with a '%' sign.",Measure,Margin % (UDF),⚠️
"Provide format string for 'Date' columnsColumns of type ""DateTime"" that have ""Date"" in their names should be formatted.",Column,'DimCustomer'[BirthDate],⚠️
"Provide format string for 'Date' columnsColumns of type ""DateTime"" that have ""Date"" in their names should be formatted.",Column,'DimCustomer'[DateFirstPurchase],⚠️
"Provide format string for 'Date' columnsColumns of type ""DateTime"" that have ""Date"" in their names should be formatted.",Column,'DimDate'[DateKey],⚠️


In [10]:
# Proof that the model still works: the cleanup removed bytes, not answers.

fabric.evaluate_dax(DATASET, workspace=WORKSPACE, dax_string="""
EVALUATE
SUMMARIZECOLUMNS(
    DimProductCategory[ProductCategoryName],
    "Total Sales", [Total Sales],
    "Margin %", [Margin %]
)
""")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 42, Finished, Available, Finished, False)

,DimProductCategory[ProductCategoryName],[Total Sales],[Margin %]
0,Audio,40462328.99,0.560907
1,TV and Video,451961041.18,0.54651
2,Computers,661417240.29,0.570522
3,Cameras and camcorders,660980251.78,0.614596
4,Cell phones,184092827.68,0.558691
5,"Music, Movies and Audio Books",33351133.62,0.610064
6,Games and Toys,36790218.694,0.537229
7,Home Appliances,915143878.87,0.55864
